# Zadaća 12

Preuzmite [dataset diabetes](https://www.kaggle.com/datasets/hasibur013/diabetes-dataset). Istrenirajte sve ansambl modele koji su obrađeni na 12. predavanju. Za stacking metodu koristite tri klasifikatora po želji. Slučajne šume trenirajte sa 50 stabala, a u Boost metodama zadajte 100 za broj baznih modela. Ispišite koje su točnosti sva četiri modela. Koristite train_test_split, zbog jednostavnosti.

In [48]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn import tree, naive_bayes, ensemble
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

In [49]:
df = pd.read_csv("diabetes_dataset.csv", low_memory=False)

In [50]:
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [51]:
df.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

Dataset ima jednu klasifikacijsku varijablu i to je `Outcome`. Ne postoje vrijednosti koje nedostaju.

In [52]:
target = df['Outcome']
features = df.drop(columns=['Outcome'])

## Stacking

In [53]:
base_estimators = [
    ('KNN', KNeighborsClassifier()),
    ('DecisionTreeClassifier', tree.DecisionTreeClassifier()),
    ('GaussianNB', naive_bayes.GaussianNB()),
]

ensemble_model = ensemble.VotingClassifier(estimators=base_estimators)
print("Score:", cross_val_score(ensemble_model, features, target, cv=5))

Score: [0.72077922 0.73376623 0.75974026 0.79738562 0.76470588]


## Slučajne šume

In [54]:
features_train, features_test, target_train, target_test = train_test_split(features, target, test_size=0.2, random_state=42)

In [55]:
clf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=None)
clf.fit(features_train, target_train)
predictions = clf.predict(features_test)
print("Accuracy: ", accuracy_score(target_test, predictions))

Accuracy:  0.7207792207792207


## Boosting

In [56]:
gb = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=42)
ada = AdaBoostClassifier(estimator=DecisionTreeClassifier(max_depth=1, random_state=42), n_estimators=100, learning_rate=0.5, random_state=42)
gb.fit(features_train, target_train)
ada.fit(features_train, target_train)

for name, model in [('GradientBoostingClassifier', gb), ('AdaBoostClassifier', ada)]:
    predictions = model.predict(features_test)
    acc = accuracy_score(predictions, target_test)
    cm = confusion_matrix(predictions, target_test)
    print(f"{name} - Accuracy: {acc:0.3f}")
    print(f"{name} - Confusion Matrix:\n{cm}\n")

GradientBoostingClassifier - Accuracy: 0.747
GradientBoostingClassifier - Confusion Matrix:
[[78 18]
 [21 37]]

AdaBoostClassifier - Accuracy: 0.773
AdaBoostClassifier - Confusion Matrix:
[[83 19]
 [16 36]]

